# 🤺 Fencing AI — Train the model (free, in your browser)

Runs the whole pipeline on **Google Colab** — no PC needed, works from a tablet.

**Pipeline:** download FIE videos → MediaPipe pose → AI labels → train → export `sabre.onnx`.

**Cost options (pick one in Step 3):**
| Provider | Cost | Quality | Sign-up |
|---|---|---|---|
| **NVIDIA NIM** | **$0** (free tier) | Good | [build.nvidia.com](https://build.nvidia.com) |
| **OpenRouter** | **$0** (free models) | Good | [openrouter.ai](https://openrouter.ai) |
| **Anthropic Haiku** | ~$25 / 50 videos | Best | [console.anthropic.com](https://console.anthropic.com) |
| **Hand-label** | $0 | You decide | No key needed — label in `/dataset` page |

Run each cell in order (Shift+Enter).

> **Tip:** Runtime → Change runtime type → **T4 GPU** (free) for faster training, though CPU works fine too.

## 1. Get the code

If your GitHub repo is **private**, create a token at https://github.com/settings/tokens (classic, `repo` scope) and paste it when prompted. If it's public, just press Enter.

In [ ]:
from getpass import getpass
import os

REPO = "crazymelol/thementalsport.com"
token = getpass("GitHub token (press Enter if repo is public): ").strip()

url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
!rm -rf thementalsport.com
!git clone --depth 1 --branch claude/new-project-BzhH4 {url}
os.chdir("thementalsport.com/fencing-ai/backend")
print("\nNow in:", os.getcwd())

## 2. Install dependencies (~2 min)

In [ ]:
!pip install -q -r requirements.txt onnxruntime
print("✓ dependencies installed")

## 3. Choose your labeling provider

**Option A — NVIDIA NIM (free, no credit card)**
1. Go to [build.nvidia.com](https://build.nvidia.com), sign up (free)
2. Click any model → **Get API Key** → copy it

**Option B — OpenRouter (free models, no credit card)**
1. Go to [openrouter.ai](https://openrouter.ai), sign up (free)
2. Settings → Keys → Create key → copy it

**Option C — Anthropic (best label quality, ~$25/50 videos)**
1. Go to [console.anthropic.com](https://console.anthropic.com/settings/keys)
2. Create key → copy it

**Option D — Skip AI labeling ($0, label by hand)**
- Set `PROVIDER = "skip"` below — dataset builds pose data only; label in the `/dataset` page.

In [ ]:
# ── Edit these two lines ──────────────────────────────────────────────────────
PROVIDER = "nvidia"        # nvidia | openrouter | anthropic | skip
API_KEY  = getpass(f"API key for {PROVIDER} (leave blank if skip): ").strip()
# ─────────────────────────────────────────────────────────────────────────────

_key_env = {
    "anthropic":  "ANTHROPIC_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
    "nvidia":     "NVIDIA_API_KEY",
}
_default_model = {
    "anthropic":  "claude-haiku-4-5-20251001",
    "openrouter": "meta-llama/llama-3.2-11b-vision-instruct:free",
    "nvidia":     "meta/llama-3.2-11b-vision-instruct",
}

if PROVIDER != "skip":
    os.environ["ANALYZER_PROVIDER"] = PROVIDER
    os.environ[_key_env[PROVIDER]]  = API_KEY
    os.environ.setdefault("ANALYZER_MODEL", _default_model[PROVIDER])
    print(f"✓ provider={PROVIDER}, model={os.environ['ANALYZER_MODEL']}")
else:
    print("✓ skip mode — pose-only dataset, label manually in /dataset")

## 4. Paste your FIE bout URLs

From the official channel: https://www.youtube.com/@FIEvideo — open sabre bouts and copy the URLs. Add as many as you like (50+ recommended).

In [ ]:
urls = """
https://www.youtube.com/watch?v=REPLACE_ME_1
https://www.youtube.com/watch?v=REPLACE_ME_2
""".strip()

with open("dataset/urls.txt", "w") as f:
    f.write(urls)
print("Saved", len([u for u in urls.splitlines() if u.strip()]), "URLs")

## 5. Build the dataset (AI auto-labels every video)

Downloads each video, runs pose estimation, labels actions. Resumable — re-run to add more videos.

*If you chose `skip` in Step 3, this still runs and extracts pose data — just without AI labels. Go label in the `/dataset` page afterwards.*

In [ ]:
!python -m dataset.build_dataset dataset/urls.txt

## 6. Train the model

In [ ]:
!python -m training.train_action_classifier --epochs 30

## 7. Export to ONNX for the live judge

In [ ]:
!python -m training.export_onnx --out sabre.onnx
print("\nFiles produced: sabre.onnx + sabre.json")

## 8. Download the trained model

Save these two files, then drop them into `fencing-ai/frontend/public/models/` (commit + push, or upload in your host's dashboard). Reload `/live` — the badge flips to **AI MODEL**.

In [ ]:
from google.colab import files
files.download("sabre.onnx")
files.download("sabre.json")